In [16]:
import tifffile
import imageio
from pathlib import Path
import imageio.v3 as iio
import numpy as np
from matplotlib.colors import to_rgb


In [12]:
input_stack = Path(r"y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\stack_outputs_tracked\A1_Neurons_NXD8_labels_tracked.tif")


In [18]:
DOWNSAMPLE = 8          # 13760 px -> 1720 px per side; a full-res gif frame would be ~190 MB
CHUNK_ROWS = 1024       # full-res rows read from disk at once (multiple of DOWNSAMPLE)
LABEL_COLOURS = {1: "deeppink", 2: "dodgerblue"}  # channel -> colour; channel 0 is brightfield
LABEL_OPACITY = 0.5
FPS = 10

stack = tifffile.memmap(input_stack, mode="r")  # (T, C, Y, X) memory-mapped, nothing is loaded until sliced
n_t, n_c, size_y, size_x = stack.shape
out_y, out_x = size_y // DOWNSAMPLE, size_x // DOWNSAMPLE
print(stack.shape, "->", (n_t, out_y, out_x))


def block_mean(plane):
    h, w = plane.shape
    return plane.reshape(h // DOWNSAMPLE, DOWNSAMPLE, w // DOWNSAMPLE, DOWNSAMPLE).mean(axis=(1, 3))


def downsample_timepoint(t):
    """Read one timepoint in row strips and return the downsampled brightfield plus,
    per label channel, the fraction of each output pixel covered by a label (0-1)."""
    brightfield = np.zeros((out_y, out_x), np.float32)
    coverage = {c: np.zeros((out_y, out_x), np.float32) for c in LABEL_COLOURS}
    for y0 in range(0, out_y * DOWNSAMPLE, CHUNK_ROWS):
        y1 = min(y0 + CHUNK_ROWS, out_y * DOWNSAMPLE)
        rows = slice(y0 // DOWNSAMPLE, y1 // DOWNSAMPLE)
        brightfield[rows] = block_mean(np.asarray(stack[t, 0, y0:y1, :out_x * DOWNSAMPLE]))
        for c in LABEL_COLOURS:
            # tracked ids and unlinked objects (-1) are both labels; 0 is background
            coverage[c][rows] = block_mean(stack[t, c, y0:y1, :out_x * DOWNSAMPLE] != 0)
    return brightfield, coverage


(16, 3, 13760, 13760) -> (16, 1720, 1720)


In [14]:
brightfields, coverages = [], []
for t in range(n_t):
    print(f"Reading timepoint {t + 1}/{n_t}")
    bf, cov = downsample_timepoint(t)
    brightfields.append(bf)
    coverages.append(cov)


Reading timepoint 1/16
Reading timepoint 2/16
Reading timepoint 3/16
Reading timepoint 4/16
Reading timepoint 5/16
Reading timepoint 6/16
Reading timepoint 7/16
Reading timepoint 8/16
Reading timepoint 9/16
Reading timepoint 10/16
Reading timepoint 11/16
Reading timepoint 12/16
Reading timepoint 13/16
Reading timepoint 14/16
Reading timepoint 15/16
Reading timepoint 16/16


In [19]:
# one contrast range for every frame so the brightfield doesn't flicker; ignore empty (0) pixels
bf_all = np.stack(brightfields)
low, high = np.percentile(bf_all[bf_all > 0], [1, 99.5])

frames = []
for bf, cov in zip(brightfields, coverages):
    rgb = np.repeat(np.clip((bf - low) / (high - low), 0, 1)[..., None], 3, axis=-1)
    for c, colour in LABEL_COLOURS.items():
        alpha = LABEL_OPACITY * cov[c][..., None]
        rgb = rgb * (1 - alpha) + np.array(to_rgb(colour)) * alpha
    frames.append((rgb * 255).astype(np.uint8))

output_dir = Path(r"C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn")  # next to this notebook; gitignored
output_dir.mkdir(exist_ok=True)
output_gif = output_dir / f"{input_stack.stem}_overlay_{FPS}fps.gif"
iio.imwrite(output_gif, frames, duration=1000 / FPS, loop=0)
print(output_gif.resolve())


C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\A1_Neurons_NXD8_labels_tracked_overlay_10fps.gif


In [9]:
input_dir = Path(r"C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif")
pngs =   sorted(list(input_dir.rglob("*.png")))
print(len(pngs))


5


In [10]:
colour_images = []

import random

# for image in random.sample(pngs, min(10, len(pngs))):
for image in pngs:
    print(image)
    img = iio.imread(image)

    # Remove alpha channel if PNG is RGBA
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[..., :3]

    colour_images.append(img)

max_y = max(img.shape[0] for img in colour_images)
max_x = max(img.shape[1] for img in colour_images)

images = []

for img in colour_images:
    pad_y = max_y - img.shape[0]
    pad_x = max_x - img.shape[1]

    padded = np.pad(
        img,
        ((0, pad_y), (0, pad_x), (0, 0)),  # don't pad RGB dimension
        mode="constant",
        constant_values=0
    )

    images.append(padded)



C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif\trackedlink03.png
C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif\trackedlink05.png
C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif\trackedlink07.png
C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif\trackedlink09.png
C:\Users\taylorhearn\Dropbox\Shared_With_Me\single_slice\togif\trackedlink11.png


In [11]:
fpss= [1,2,5]
for fps in fpss:
    imageio.mimsave(
        f"{str(input_dir)}/gif_{fps}fps.gif",
        images,
        fps=fps
    )